In [3]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime

In [4]:
pdf_folder_location = "data/raw/Amazon-2025-Annual-Report.pdf"

In [5]:
loader=PyPDFLoader(pdf_folder_location)
documents=loader.load()

    

In [6]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000,
    chunk_overlap=150
)

In [7]:
chunks = loader.load_and_split(text_splitter)

In [11]:
print(chunks[0].page_content)

ANNUAL REPORT
2 0 2 5


In [12]:
tesla_collection = 'tesla-Amazon-2025'

In [13]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [14]:

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Simran Kumar\AppData\Local\Temp\ipykernel_7480\2427895705.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [15]:
chromadb_client = chromadb.PersistentClient(
    path="./data/vector_store"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [16]:
chromadb_client.heartbeat()

1781250390498092300

In [17]:
vectorstore = Chroma(
    collection_name=tesla_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [18]:
i = 0 # Initialize the starting index for the chunks

while i < len(chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(30)

In [22]:
collection = chromadb_client.get_collection(tesla_collection)

In [23]:
collection.peek()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['text_0',
  'text_1',
  'text_2',
  'text_3',
  'text_4',
  'text_5',
  'text_6',
  'text_7',
  'text_8',
  'text_9'],
 'embeddings': array([[-0.02464059,  0.04807854, -0.00068172, ...,  0.00178862,
          0.00688634, -0.04186994],
        [ 0.04634272,  0.04858634, -0.01857388, ..., -0.01968449,
          0.02431422, -0.03767077],
        [ 0.02615474,  0.04067658, -0.03622886, ...,  0.01703247,
         -0.0287286 , -0.03533861],
        ...,
        [ 0.01841464,  0.09604206, -0.04143979, ..., -0.00326636,
         -0.02553031, -0.05343892],
        [ 0.04315913,  0.07416114, -0.01405658, ..., -0.00959689,
         -0.01706993, -0.02990051],
        [ 0.02093196,  0.10357454, -0.03170317, ..., -0.00064259,
          0.04229002, -0.04087727]]),
 'documents': ['ANNUAL REPORT\n2 0 2 5',
  'Dear Shareholders:\nWhen I graduated from college, I wanted to be a sportscaster. After sending my resume reel to many small\nmarkets around the U.S., and only getting two nibbles, I sett

In [24]:
collection.get(
    ids=['text_999']
)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'data': None,
 'metadatas': [],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [28]:
vectorstore_persisted = Chroma(
    collection_name=tesla_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [29]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [30]:
user_query="What are the major risks mentioned by Tesla?"

In [31]:
retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='text_23', metadata={'author': 'T&C Composition', 'creationdate': '2022-02-14T21:08:55-06:00', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': '26-3815-1_3', 'moddate': '2026-04-07T16:34:29-05:00', 'page': 17, 'page_label': '18', 'producer': 'Adobe PDF Library 15.0', 'source': 'data/raw/Amazon-2025-Annual-Report.pdf', 'subject': 'Annual Report', 'title': 'Amazon.com, Inc', 'total_pages': 92, 'trapped': '/False'}, page_content='quality issues. In addition, profitability or other intended benefits, if any, in our newer activities (including development and \nadoption of automation, artificial intelligence, and machine learning technologies for customer and internal use), may not meet \nour expectations, and we may not be successful enough in these newer activities to recoup our investments in them, which \ninvestments are often significant. Failure to realize the benefits of amounts we inves

In [34]:



#---------------output prompts-----------------------
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

Rules:
1. The answer must be based only on retrieved context.
2. Every important claim must map to a retrieved source.
3. The assistant must not invent unsupported information.
4. If retrieved context is weak, answerability should be PARTIALLY_ANSWERED or 
NOT_FOUND.
5. If the question is speculative, the answer should refuse speculation

User queries will be delimited by: <Question> and </Question>.

Your answer should strictly be in the given JSON. nothing extra should be there
Expected output:
{
"answer": "",
"supporting_evidence": [],
"sources": [],
"confidence": "HIGH | MEDIUM | LOW",
"answerability": "ANSWERED | PARTIALLY_ANSWERED | NOT_FOUND"
}
"""

In [37]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

#-------------classification prompts-----------------

classification_system_prompt="""
Before retrieval or answer generation, the system should classify the user query.
Supported query types:
FACTUAL_LOOKUP
SUMMARY
COMPARISON
RISK_ANALYSIS
UNANSWERABLE_OR_SPECULATIVE
FOLLOW_UP
OTHER

Strictly follow this format to answer
{
"query_type": "",
"requires_retrieval": true,
"requires_comparison": false,
"answer_style": "",
"reasoning_summary": ""
}
"""


#==================citations======================
citation_system_prompt="""
The application must display source references with each answer.
Minimum source fields:
{
"source_file": "",
"page_number": "",
"chunk_id": "",
"snippet": ""
}
The snippet should be short enough to verify the answer quickly.
Example:
Source: tesla_10k_2024.pdf, page 12, chunk_0048
Snippet: "We face risks related to..."""

In [38]:

import os

os.environ['GROQ_API_KEY'] =os.getenv("GROQ_API_KEY")

from groq import Groq

client = Groq(api_key=os.environ['GROQ_API_KEY'])


#-----------------------------classification of query---------------


prompt = [
    {'role': 'system', 'content': classification_system_prompt},
    {'role': 'user', 'content': user_query
    }
]
try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

    


{
  "query_type": "FACTUAL_LOOKUP",
  "requires_retrieval": true,
  "requires_comparison": false,
  "answer_style": "concise list with brief explanations",
  "reasoning_summary": "The user asks for the major risks that Tesla mentions, which requires retrieving Tesla's disclosed risk factors (e.g., from its latest SEC Form 10‑K or 10‑Q). This is a factual lookup, not a comparison, so we set requires_retrieval to true and provide a concise list of the key risk categories."
}


In [41]:

# ------------------------RAG + LLM Final output---------------------


relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
        context=context_for_query,
        question=user_query
        )
    }
]
try:
    response = client.chat.completions.create(
        model=os.environ['GROQ_MODEL'],
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

#


{
  "answer": "",
  "supporting_evidence": [],
  "sources": [],
  "confidence": "LOW",
  "answerability": "NOT_FOUND"
}
